# PROYECTO 3
## ALGORITMOS BIOINSPIRADOS
**RODRIGO ROSALES DÍAZ** 

**MARÍA GAMBA SANTIBÁÑEZ**

**MATEO SERRATO ASCENCIO**


In [11]:
#Primero importamos librerias:
import numpy as np 
import matplotlib.pyplot as plt 
import operator 
import random
import math 
from deap import base, creator, gp, tools, algorithms
import functools
import sympy

In [12]:
#Creamos el conjunto de primitivas
pset = gp.PrimitiveSet("MAIN", 2)  # x, t
pset.addPrimitive(operator.add, 2)
pset.addPrimitive(operator.sub, 2)
pset.addPrimitive(operator.mul, 2)
pset.addPrimitive(operator.neg, 1)
pset.addPrimitive(math.sin, 1)
pset.addPrimitive(math.cos, 1)
pset.addPrimitive(math.exp, 1)
pset.addEphemeralConstant("rand", functools.partial(random.uniform, -1, 1))


In [13]:
#RENOMBRAMOS EL ARGUMENTO
pset.renameArguments(ARG0='x')
pset.renameArguments(ARG1='t')

#DEFINIR EL TIPO DE FITNESS (minimizar el error)
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individuo", gp.PrimitiveTree, fitness=creator.FitnessMin)

#Funciones para inicializar individuos y población
toolbox = base.Toolbox()

toolbox.register('expr', gp.genHalfAndHalf, pset = pset, min_ = 1, max_ = 3)
toolbox.register('individuo', tools.initIterate, creator.Individuo, toolbox.expr)
toolbox.register('population', tools.initRepeat,list,toolbox.individuo)

# FUNCION PARA COMPILAR LOS ÁRBOLES EN FUNCIONES EJECUTABLES
toolbox.register("compile", gp.compile, pset=pset)


c:\Users\mateo\AppData\Local\Programs\Python\Python313\Lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'FitnessMin' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
c:\Users\mateo\AppData\Local\Programs\Python\Python313\Lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'Individuo' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [14]:
def evaluar_pde(individual, points, alpha=0.1):
    """
    Evalúa qué tan bien un individuo (función u(x,t)) satisface la ecuación de calor.
    ∂u/∂t = alpha * ∂²u/∂x²
    """
    func = toolbox.compile(expr=individual)
    
    # Pequeños deltas para la diferenciación numérica
    h_x = 1e-4
    h_t = 1e-4
    
    total_residual = 0.0
    
    for x, t in points:
        try:
            # Calcular ∂u/∂t (derivada parcial respecto al tiempo)
            du_dt = (func(x, t + h_t) - func(x, t - h_t)) / (2 * h_t)
            
            # Calcular ∂²u/∂x² (segunda derivada parcial respecto a la posición)
            d2u_dx2 = (func(x + h_x, t) - 2 * func(x, t) + func(x - h_x, t)) / (h_x**2)
            
            # Calcular el residual de la EDP: ∂u/∂t - alpha * ∂²u/∂x²
            # El objetivo es que este residual sea 0
            residual = du_dt - alpha * d2u_dx2
            total_residual += residual**2
            
        except (OverflowError, ValueError):
            # Penalizar individuos que resultan en errores numéricos (ej. exp(muy grande))
            return 1e9, # Un valor de error muy alto

    # Devolver el error cuadrático medio del residual
    return total_residual / len(points),



In [15]:
# Generar los puntos de evaluación
xs = np.linspace(0, 1, 10)
ts = np.linspace(0, 1, 10)
points = [(x, t) for x in xs for t in ts]


In [16]:
# Registrar la función de evaluación con los puntos fijos
toolbox.register("evaluate", functools.partial(evaluar_pde, points=points))

In [17]:
# ALGORITMO EVOLUTIVO

# Registrar operadores evolutivos
toolbox.register("mate", gp.cxOnePoint)
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr, pset=pset)
toolbox.register("select", tools.selTournament, tournsize=3)
Poblacion = toolbox.population(n=100)
hof = tools.HallOfFame(1)

stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

algorithms.eaSimple(Poblacion, toolbox, 0.5, 0.05, 10, stats=stats, halloffame=hof, verbose=True)

# %%
print("\nMejor individuo:")
print(hof[0])

# Visualizar con sympy
x, t = sympy.symbols("x t")

replacements = {
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
    "neg": lambda a: -a,
    "sin": sympy.sin,
    "cos": sympy.cos,
    "exp": sympy.exp
}

expr_str = str(hof[0])
expr_sympy = sympy.sympify(expr_str, locals=replacements)
print("\nExpresión simplificada:\n", sympy.simplify(expr_sympy))

gen	nevals	avg     	std    	min	max    
0  	100   	0.644979	1.26491	0  	8.85488
1  	48    	3.38287 	32.265 	0  	324.285
2  	55    	0.0454121	0.398377	0  	4      
3  	46    	0.0146165	0.0882663	0  	0.718346
4  	60    	0.00884301	0.0650779	0  	0.593277
5  	42    	0.396562  	3.91027  	0  	39.3027 
6  	61    	0.00733275	0.0431046	0  	0.308699
7  	52    	0.00893739	0.0603402	0  	0.463986
8  	49    	0.0155593 	0.105222 	0  	1       
9  	58    	0.0263472 	0.262152 	0  	2.63472 
10 	42    	0.0157025 	0.113686 	0  	1       

Mejor individuo:
neg(cos(0.4909176776145394))

Expresión simplificada:
 -0.8819006043079097
